# 気象庁 過去の気象データ 一括ダウンロード

[観測所ファインダー](https://awg-yk.github.io/weather-station-finder/) で選んだ
**地点リスト**を使って、気象庁の過去の気象データをまとめて取得するノートブックです。

## 使い方
1. 下のセルを実行（▶ボタン）すると**入力フォーム**が表示されます
2. 地点リストを渡す（どちらか）:
   - **(A) 貼り付け**: ファインダーの「Colab用にコピー」を押し、フォームの「または貼付」欄に貼る（ファイル不要）
   - **(B) ファイル**: 「選択結果をCSVでダウンロード」で保存したCSVを「CSVを選択」でアップロード
3. 期間の種類・データの種類・観測項目・期間を**プルダウンで選択**
4. 「ダウンロード開始」を押すと取得が進み、最後にZIPが手元に落ちてきます

- 「連続した期間」と「特定の期間を複数年分」の2モードに対応
- **気象庁の1回あたりデータ量上限を超えないよう、期間を自動で分割**して取得します
- 取得後は地点ごとに1ファイルへ自動結合（ファイル数を削減）
- 気象庁サーバーに負荷をかけないよう、1件ずつ間隔（既定3秒）を空けて取得します


In [ ]:
# ============================================================
# 気象庁 過去の気象データ 一括ダウンロード（Google Colab・フォーム版）
#
# 観測所ファインダー( https://awg-yk.github.io/weather-station-finder/ )で
# 出力したCSVを入力に、指定した種類・項目・期間のデータをまとめて取得します。
#
# 使い方:
#   1. このセルを実行（▶）するとフォームが表示されます
#   2. 地点リストを渡す（どちらか）:
#        (A) ファインダーの「Colab用にコピー」を押し、フォームの「または貼付」欄に貼り付け
#        (B) 「選択結果をCSVでダウンロード」で保存したCSVを「CSVを選択」でアップロード
#   3. データの種類・観測項目・期間などをプルダウンで選択
#   4. 「ダウンロード開始」ボタンを押すと取得が進み、最後にZIPが手元に落ちてきます
#
# 特徴:
#   - すべてプルダウン等のフォームで選択（横スクロールで隠れる問題を解消）
#   - 「連続した期間」と「特定の期間を複数年分」の2モードに対応
#   - 取得後、地点ごとに1ファイルへ自動結合してファイル数を削減
#   - 地点番号(prec_no/block_no)はCSVから直接取得。取得済みファイルはスキップ再開
#   - 気象庁サーバーに負荷をかけないよう、1件ずつスリープを挟んで取得します
# ============================================================

!pip install -q requests ipywidgets

import calendar
import csv as csv_module
import io
import json
import re
import shutil
import time
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import parse_qs, urlparse

import requests
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

ROOT_URL = "https://www.data.jma.go.jp/risk/obsdl/index.php"
SHOW_URL = "https://www.data.jma.go.jp/risk/obsdl/show/table"
STATIONS_JSON_URL = "https://raw.githubusercontent.com/awg-yk/weather-station-finder/main/data/stations.json"

MAX_RETRIES = 3

# 集計期間: (コード, 表示名, 連続モードでの分割単位)
PERIOD_OPTIONS: List[Tuple[str, str, str]] = [
    ("9", "時別値",     "month"),
    ("1", "日別値",     "year"),
    ("2", "半旬別値",   "year"),
    ("4", "旬別値",     "year"),
    ("5", "月別値",     "year"),
    ("6", "3か月別値",  "year"),
]

# 時別値(9)用の観測項目: (コード, 表示名, カテゴリ)
ELEMENTS_HOURLY: List[Tuple[str, str, str]] = [
    ("201", "気温", "気温"),
    ("101", "降水量（前1時間）", "降水量"),
    ("301", "風向・風速", "風"),
    ("401", "日照時間（前1時間）", "日照時間"),
    ("610", "全天日射量（前1時間）", "日照時間"),
    ("501", "積雪の深さ", "積雪"),
    ("503", "降雪の深さ（前1時間）", "積雪"),
    ("605", "相対湿度", "湿度"),
    ("604", "蒸気圧", "湿度"),
    ("612", "露点温度", "湿度"),
    ("601", "現地気圧", "湿度"),
    ("602", "海面気圧", "湿度"),
    ("607", "雲量", ""),
    ("703", "天気", ""),
    ("704", "視程", ""),
]

# 日別/半旬/旬/月/3か月 共通の観測項目: (コード, 表示名, 対応期間集合, カテゴリ)
ELEMENTS_OTHER: List[Tuple[str, str, set, str]] = [
    ("201", "平均気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("202", "最高気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("203", "最低気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("204", "日最高気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("206", "日最低気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("205", "日最高気温の最低",    {"2", "4", "5", "6"}, "気温"),
    ("207", "日最低気温の最高",    {"2", "4", "5", "6"}, "気温"),
    ("101", "降水量の合計",        {"1", "2", "4", "5", "6"}, "降水量"),
    ("102", "日降水量の最大",      {"2", "4", "5", "6"}, "降水量"),
    ("401", "日照時間",            {"1", "2", "4", "5", "6"}, "日照時間"),
    ("610", "合計全天日射量",      {"1", "2", "4", "5", "6"}, "日照時間"),
    ("501", "最深積雪",            {"1", "2", "4", "5", "6"}, "積雪"),
    ("503", "降雪量の合計",        {"1", "2", "4", "5", "6"}, "積雪"),
    ("504", "降雪量日合計の最大",  {"2", "4", "5", "6"}, "積雪"),
    ("301", "平均風速",            {"1", "2", "4", "5", "6"}, "風"),
    ("302", "最大風速（風向）",    {"1", "2", "4", "5", "6"}, "風"),
    ("304", "最大瞬間風速（風向）", {"1", "2", "4", "5", "6"}, "風"),
    ("305", "最多風向",            {"1", "2", "4", "5", "6"}, "風"),
    ("605", "平均相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("606", "最小相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("604", "平均蒸気圧",          {"1", "2", "4", "5", "6"}, "湿度"),
    ("601", "平均現地気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("602", "平均海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("603", "最低海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("607", "平均雲量",            {"1", "2", "4", "5", "6"}, ""),
    ("701", "天気概況（昼）",      {"1"}, ""),
    ("702", "天気概況（夜）",      {"1"}, ""),
]

ID_COLUMN_CANDIDATES = ["観測所ID", "地点コード", "地点番号"]


@dataclass
class Station:
    station_id: str
    name: str
    prec_no: str
    block_no: str
    station_type: str
    elements: set
    status: str

    def station_num(self) -> str:
        if self.station_type == "気象官署":
            return "s" + self.block_no
        return "a" + self.block_no.zfill(4)


@dataclass
class WeatherDataPayload:
    stationNumList: List[str] = field(default_factory=list)
    aggrgPeriod: int = 1
    elementNumList: List[List[str]] = field(default_factory=list)
    interAnnualType: int = 1
    ymdList: List[str] = field(default_factory=list)  # [y1, y2, m1, m2, d1, d2]
    optionNumList: List[Any] = field(default_factory=list)
    downloadFlag: str = "true"
    rmkFlag: int = 1
    disconnectFlag: int = 1
    youbiFlag: int = 0
    fukenFlag: int = 0
    kijiFlag: int = 0
    huukouFlag: int = 0
    csvFlag: int = 1
    jikantaiFlag: int = 0
    jikantaiList: List[Any] = field(default_factory=list)
    ymdLiteral: int = 1

    def to_post_data(self) -> dict:
        data = {}
        for key, value in asdict(self).items():
            data[key] = json.dumps(value) if isinstance(value, list) else value
        return data


# ------------------------------------------------------------
# 入力CSVの読み込み
# ------------------------------------------------------------
def parse_prec_block_from_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    if not url:
        return None, None
    try:
        q = parse_qs(urlparse(url).query)
        return q.get("prec_no", [None])[0], q.get("block_no", [None])[0]
    except Exception:
        return None, None


def parse_elements_cell(cell: str) -> set:
    if not cell:
        return set()
    return {p.strip() for p in re.split(r"[\/／,、]", cell) if p.strip()}


def load_master_fallback() -> Dict[str, dict]:
    cache = Path("stations_master.json")
    if not cache.exists():
        resp = requests.get(STATIONS_JSON_URL, timeout=30)
        resp.raise_for_status()
        cache.write_bytes(resp.content)
    raw = json.loads(cache.read_text(encoding="utf-8"))
    return {str(s["id"]): s for s in raw["stations"]}


def find_id_column(fieldnames: List[str]) -> Optional[str]:
    for cand in ID_COLUMN_CANDIDATES:
        if cand in fieldnames:
            return cand
    return None


def read_stations_from_text(text: str) -> List[Station]:
    text = text.lstrip("﻿")  # 貼り付け時に残るBOMを除去（列名の頭に付くと列検出に失敗するため）
    reader = csv_module.DictReader(io.StringIO(text))
    fields = reader.fieldnames or []
    id_col = find_id_column(fields)
    if id_col is None:
        raise ValueError(f"CSVに地点IDの列（{'／'.join(ID_COLUMN_CANDIDATES)}）が見つかりません。列: {fields}")
    rows = list(reader)

    has_url = "気象庁ページURL" in fields
    master = None if has_url else load_master_fallback()

    out: List[Station] = []
    for row in rows:
        sid = (row.get(id_col) or "").strip()
        if not sid:
            continue
        name = (row.get("地点名") or "").strip()
        stype = (row.get("種別") or "").strip()
        status = (row.get("状態") or "").strip()
        elems = parse_elements_cell(row.get("観測要素", ""))

        prec = block = None
        if has_url:
            prec, block = parse_prec_block_from_url(row.get("気象庁ページURL", ""))
        if (not prec or not block) and master is not None:
            m = master.get(sid)
            if m:
                prec, block = m.get("precNo"), m.get("blockNo")
                if not stype:
                    stype = m.get("stationType", "")
        if not prec or not block:
            print(f"  [スキップ] {name or sid}: 地点番号を特定できません")
            continue
        if not stype:
            stype = "気象官署" if len(block) >= 5 and block.startswith("47") else "アメダス"
        out.append(Station(sid, name or sid, prec, block, stype, elems, status))
    return out


# ------------------------------------------------------------
# 期間の分割（連続モードのみ。複数年モードは分割せず1リクエスト）
# ------------------------------------------------------------
def date_chunks(start: date, end: date, unit: str):
    cur = start
    while cur <= end:
        if unit == "month":
            if cur.month == 12:
                last = date(cur.year, 12, 31)
            else:
                last = date(cur.year, cur.month + 1, 1) - timedelta(days=1)
            chunk_end = min(last, end)
            nxt_month = cur.month + 1
            nxt_year = cur.year + (1 if nxt_month > 12 else 0)
            nxt_month = 1 if nxt_month > 12 else nxt_month
            nxt = date(nxt_year, nxt_month, 1)
        else:
            chunk_end = min(date(cur.year, 12, 31), end)
            nxt = date(cur.year + 1, 1, 1)
        yield cur, chunk_end
        cur = nxt


# ------------------------------------------------------------
# 1リクエストのデータ量が気象庁の上限を超えないよう分割する
#   気象庁の判定式（top.2.1.js より）:
#     地点数 × 項目数 × 期間の点数(nOfPr) × 重み ≤ seigen(=44000)
#   本ツールは1リクエスト=1地点なので、項目数 × 点数 × 重み ≤ 上限 になるよう分割する。
# ------------------------------------------------------------
SEIGEN = 44000
VOLUME_LIMIT = 40000  # 44000に対して余裕を持たせた実効上限


def _safe_date(y: int, m: int, d: int) -> date:
    last = calendar.monthrange(y, m)[1]
    return date(y, m, min(d, last))


def _days_between(y1, m1, d1, y2, m2, d2) -> int:
    return abs((_safe_date(y2, m2, d2) - _safe_date(y1, m1, d1)).days) + 1


def count_periods(aggrg_type: int, inter: int, y1, y2, m1, m2, d1, d2) -> int:
    """気象庁 countPrNum を移植。期間の点数(nOfPr)を返す。ymd=[y1,y2,m1,m2,d1,d2]。"""
    if inter == 1:  # 連続した期間
        if aggrg_type in (1, 8, 9):
            diff = _days_between(y1, m1, d1, y2, m2, d2)
            if aggrg_type == 9:
                diff *= 24
        elif aggrg_type in (2, 4):
            sub = 6 if aggrg_type == 2 else 3
            diff = abs((y2 - y1) * 12 * sub + (m2 - m1) * sub + (d2 - d1)) + 1
        elif aggrg_type in (5, 6):
            diff = abs(y2 * 12 + m2 - y1 * 12 - m1) + 1
        else:
            diff = _days_between(y1, m1, d1, y2, m2, d2)
    else:  # 特定の期間を複数年分
        if aggrg_type in (1, 8, 9):
            dt1 = _safe_date(y1, m2, d2)
            dt2 = _safe_date(y1, m1, d1)
            if dt1 < dt2:
                dt1 = _safe_date(y1 + 1, m2, d2)
            diff_day = abs((dt1 - dt2).days) + 1
            diff_year = abs(y2 - y1) + 1
            diff = diff_year * diff_day
            if aggrg_type == 9:
                diff *= 24
        elif aggrg_type in (2, 4):
            sub = 6 if aggrg_type == 2 else 3
            yd = abs(y1 - y2) + 1
            if m1 < m2 or (m1 == m2 and d1 <= d2):
                md = abs((m2 - m1) * sub + (d2 - d1)) + 1
            else:
                md = abs(12 * sub - ((m1 - m2) * sub + (d1 - d2))) + 1
            diff = yd * md
        elif aggrg_type in (5, 6):
            yd = abs(y1 - y2) + 1
            md = (m2 - m1 + 1) if m1 <= m2 else (12 - (m1 - m2) + 1)
            diff = yd * md
        else:
            diff = abs(y2 - y1) + 1
    return int(diff)


def build_plan(inter_type, aggrg_type, n_el, y1, m1, d1, y2, m2, d2, chunk_unit):
    """各リクエストが上限内に収まる計画を作る。戻り値: [(inter, [Y1,Y2,M1,M2,D1,D2], 名前suffix), ...]"""
    weight = 1.5 if aggrg_type == 8 else 1.0

    def cost(inter, Y1, Y2, M1, M2, D1, D2):
        return n_el * weight * count_periods(aggrg_type, inter, Y1, Y2, M1, M2, D1, D2)

    def split_cont(cs, ce):
        # 連続期間を、上限を超えるなら日数で二分して収める
        if cost(1, cs.year, ce.year, cs.month, ce.month, cs.day, ce.day) <= VOLUME_LIMIT or cs >= ce:
            return [("1", [cs.year, ce.year, cs.month, ce.month, cs.day, ce.day], f"{cs.isoformat()}_{ce.isoformat()}")]
        mid = cs + (ce - cs) // 2
        return split_cont(cs, mid) + split_cont(mid + timedelta(days=1), ce)

    specs = []
    if inter_type == "1":
        d1c = min(d1, calendar.monthrange(y1, m1)[1])
        d2c = min(d2, calendar.monthrange(y2, m2)[1])
        for cs, ce in date_chunks(date(y1, m1, d1c), date(y2, m2, d2c), chunk_unit):
            specs.extend(split_cont(cs, ce))
    else:
        d1c = min(d1, calendar.monthrange(2000, m1)[1])
        d2c = min(d2, calendar.monthrange(2000, m2)[1])
        per_year = cost(2, 2000, 2000, m1, m2, d1c, d2c)  # 1年分の量
        if per_year <= VOLUME_LIMIT:
            batch = max(1, int(VOLUME_LIMIT // per_year))
            y = y1
            while y <= y2:
                by2 = min(y + batch - 1, y2)
                suffix = f"{y}-{by2}_各年{m1:02d}{d1c:02d}-{m2:02d}{d2c:02d}"
                specs.append(("2", [y, by2, m1, m2, d1c, d2c], suffix))
                y = by2 + 1
        else:
            # 1年分でも上限超 → 各年を連続期間として月/年分割
            for Y in range(y1, y2 + 1):
                ys = min(d1, calendar.monthrange(Y, m1)[1])
                ye = min(d2, calendar.monthrange(Y, m2)[1])
                for cs, ce in date_chunks(date(Y, m1, ys), date(Y, m2, ye), chunk_unit):
                    specs.extend(split_cont(cs, ce))
    return specs


# ------------------------------------------------------------
# 取得（リトライ＋エラーHTML検出つき）
# ------------------------------------------------------------
def looks_like_csv(content: bytes) -> bool:
    head = content[:200].lstrip()
    if head[:1] == b"<":
        return False
    return len(content) > 0


def fetch_data(session, station_num, ymd, aggrg_period, elements, inter_annual_type, sleep_sec):
    payload = WeatherDataPayload(
        stationNumList=[station_num],
        aggrgPeriod=int(aggrg_period),
        elementNumList=[[code, ""] for code in elements],
        interAnnualType=int(inter_annual_type),
        ymdList=[str(x) for x in ymd],
    )
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.post(SHOW_URL, data=payload.to_post_data(),
                                headers={"Referer": ROOT_URL}, timeout=60)
            resp.raise_for_status()
            if not looks_like_csv(resp.content):
                raise RuntimeError("CSV以外の応答（混雑またはデータ量超過の可能性）")
            return resp.content
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(sleep_sec * attempt)
    raise last_err


def save_csv(content: bytes, output_path: Path) -> None:
    try:
        output_path.write_text(content.decode("cp932"), encoding="utf-8-sig", newline="")
    except UnicodeDecodeError:
        output_path.write_bytes(content)


# ------------------------------------------------------------
# 地点ごとに複数チャンクを1ファイルへ結合
#   気象庁CSVは先頭に数行のヘッダー（時刻・地点名・項目名など）があり、
#   データ行は「年で始まる行」。先頭ファイルのヘッダー＋全ファイルのデータ行を連結する。
# ------------------------------------------------------------
def merge_station_files(paths: List[Path], out_path: Path) -> None:
    header_lines: Optional[List[str]] = None
    data_all: List[str] = []
    for p in paths:
        lines = p.read_text(encoding="utf-8-sig").splitlines()
        di = next((k for k, ln in enumerate(lines) if ln[:1].isdigit()), len(lines))
        if header_lines is None:
            header_lines = lines[:di]
        data_all.extend(lines[di:])
    out_lines = (header_lines or []) + data_all
    out_path.write_text("\r\n".join(out_lines) + "\r\n", encoding="utf-8-sig", newline="")


# ============================================================
# フォームUI（ipywidgets）
# ============================================================
CODE_TO_CATEGORY = {}
for v, lbl, cat in ELEMENTS_HOURLY:
    CODE_TO_CATEGORY[("9", v)] = cat
for v, lbl, kikan, cat in ELEMENTS_OTHER:
    for k in kikan:
        CODE_TO_CATEGORY[(k, v)] = cat

_this_year = date.today().year
_years = [str(y) for y in range(_this_year, 1899, -1)]
_months = [str(m) for m in range(1, 13)]
_days = [str(d) for d in range(1, 32)]

BOX_WIDTH = "560px"
_row = lambda *w: widgets.HBox(list(w))

style = {"description_width": "96px"}
LW = widgets.Layout(width="150px")

up = widgets.FileUpload(accept=".csv", multiple=False,
                        description="CSVを選択", layout=widgets.Layout(width="220px"))
paste_area = widgets.Textarea(
    placeholder="観測所ファインダーで「Colab用にコピー」を押し、ここに貼り付け（Ctrl+V）。ファイルのアップロードは不要です。",
    description="または貼付", style=style,
    layout=widgets.Layout(width=BOX_WIDTH, height="90px"))
mode_dd = widgets.RadioButtons(
    options=[("連続した期間で表示する", "1"), ("特定の期間を複数年分、表示する", "2")],
    value="1", description="期間の種類", style=style, layout=widgets.Layout(width=BOX_WIDTH))
period_dd = widgets.Dropdown(
    options=[(lbl, code) for code, lbl, _u in PERIOD_OPTIONS],
    value="1", description="データの種類", style=style, layout=widgets.Layout(width="300px"))
elem_sel = widgets.SelectMultiple(
    options=[], description="観測項目", style=style,
    layout=widgets.Layout(width=BOX_WIDTH, height="150px"))

syear = widgets.Dropdown(options=_years, value="2023", layout=LW)
smonth = widgets.Dropdown(options=_months, value="1", layout=LW)
sday = widgets.Dropdown(options=_days, value="1", layout=LW)
eyear = widgets.Dropdown(options=_years, value=str(_this_year), layout=LW)
emonth = widgets.Dropdown(options=_months, value="12", layout=LW)
eday = widgets.Dropdown(options=_days, value="31", layout=LW)

period_desc = widgets.HTML()
start_row = _row(widgets.Label("開始", layout=widgets.Layout(width="40px")), syear, smonth, sday)
end_row = _row(widgets.Label("終了", layout=widgets.Layout(width="40px")), eyear, emonth, eday)

sleep_dd = widgets.Dropdown(options=[("3秒（推奨）", 3.0), ("4秒", 4.0), ("5秒", 5.0), ("10秒", 10.0)],
                            value=3.0, description="取得間隔", style=style, layout=widgets.Layout(width="240px"))
merge_cb = widgets.Checkbox(value=True, description="地点ごとに1ファイルへ結合する（ファイル数を減らす）",
                            indent=False)
run_btn = widgets.Button(description="ダウンロード開始", button_style="success",
                         layout=widgets.Layout(width="200px"))
out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="6px",
                                            max_height="420px", overflow="auto"))


def refresh_elements(*_):
    pc = period_dd.value
    if pc == "9":
        opts = [(lbl, v) for v, lbl, cat in ELEMENTS_HOURLY]
    else:
        opts = [(lbl, v) for v, lbl, kikan, cat in ELEMENTS_OTHER if pc in kikan]
    elem_sel.options = opts
    # 既定で気温・降水を選択
    default = [v for (lbl, v) in opts if ("気温" in lbl or "降水" in lbl)][:2]
    elem_sel.value = tuple(default) if default else (opts[0][1],) if opts else tuple()


def refresh_desc(*_):
    if mode_dd.value == "1":
        period_desc.value = ("<span style='color:#555'>▼ <b>連続した期間</b>：開始日から終了日まで通しで取得します。</span>")
        start_row.children[0].value = "開始"
        end_row.children[0].value = "終了"
    else:
        period_desc.value = ("<span style='color:#555'>▼ <b>特定の期間を複数年分</b>：各年の「開始（月・日）〜終了（月・日）」を、"
                             "開始年〜終了年の各年について取得します（年の値が年範囲、月日が毎年の対象期間）。</span>")
        start_row.children[0].value = "開始"
        end_row.children[0].value = "終了"


period_dd.observe(refresh_elements, names="value")
mode_dd.observe(refresh_desc, names="value")
refresh_elements()
refresh_desc()


def read_uploaded_csv_text() -> Optional[Tuple[str, str]]:
    """(filename, text) を返す。未アップロードなら None。"""
    val = up.value
    if not val:
        return None
    if isinstance(val, dict):           # ipywidgets 7 系
        item = next(iter(val.values()))
        content = item["content"]
        fname = item["metadata"]["name"]
    else:                                # ipywidgets 8 系（タプル）
        item = val[0]
        content = item["content"]
        fname = item["name"]
    raw = bytes(content)
    for enc in ("utf-8-sig", "cp932"):
        try:
            return fname, raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return fname, raw.decode("utf-8", errors="replace")


def on_run(_btn):
    run_btn.disabled = True
    out.clear_output()
    with out:
        try:
            pasted = paste_area.value.strip()
            if pasted:
                csv_text = pasted
                print("貼り付けられたデータを使用します。")
            else:
                uploaded = read_uploaded_csv_text()
                if uploaded is None:
                    print("地点リストを、貼り付け欄に貼るか、CSVを選択してください。")
                    return
                _, csv_text = uploaded
            stations = read_stations_from_text(csv_text)
            if not stations:
                print("有効な地点がCSVから読み取れませんでした。")
                return

            inter_type = mode_dd.value
            period_code = period_dd.value
            period_label = dict((c, l) for c, l, _u in PERIOD_OPTIONS)[period_code]
            chunk_unit = dict((c, u) for c, _l, u in PERIOD_OPTIONS)[period_code]
            element_codes = list(elem_sel.value)
            if not element_codes:
                print("観測項目を1つ以上選択してください。")
                return
            sel_cats = {CODE_TO_CATEGORY.get((period_code, c), "") for c in element_codes}
            sel_cats.discard("")
            sleep_sec = float(sleep_dd.value)

            y1, m1, d1 = int(syear.value), int(smonth.value), int(sday.value)
            y2, m2, d2 = int(eyear.value), int(emonth.value), int(eday.value)
            aggrg_type = int(period_code)

            # 期間の妥当性チェック
            if inter_type == "2":
                if y1 > y2:
                    print("開始年は終了年以前にしてください。")
                    return
            else:
                if date(y1, m1, min(d1, calendar.monthrange(y1, m1)[1])) > date(y2, m2, min(d2, calendar.monthrange(y2, m2)[1])):
                    print("開始日は終了日以前にしてください。")
                    return

            # リクエスト計画（各リクエストが気象庁の上限内に収まるよう自動分割）
            plan = build_plan(inter_type, aggrg_type, len(element_codes), y1, m1, d1, y2, m2, d2, chunk_unit)
            n_per_station = len(plan)

            n_requests = len(stations) * n_per_station
            est_min = n_requests * sleep_sec / 60.0
            discontinued = [s for s in stations if s.status and s.status != "現役"]
            mismatch = [s.name for s in stations if s.elements and sel_cats and not (sel_cats & s.elements)]

            print("── 設定内容 ──")
            print(f"  期間の種類   : {'特定の期間を複数年分' if inter_type == '2' else '連続した期間'}")
            print(f"  データの種類 : {period_label}")
            print(f"  観測項目     : {', '.join(element_codes)}")
            if inter_type == "2":
                print(f"  期間         : 各年 {m1}/{d1} 〜 {m2}/{d2} を {y1}年〜{y2}年")
            else:
                print(f"  期間         : {y1}/{m1}/{d1} 〜 {y2}/{m2}/{d2}（{'1か月' if chunk_unit=='month' else '1年'}ごと）")
            print(f"  対象地点数   : {len(stations)} 地点")
            print(f"  リクエスト数 : 約 {n_requests} 回（1地点あたり {n_per_station} 分割・間隔 {sleep_sec}秒 → 推定 約 {est_min:.1f} 分）")
            if n_per_station > 1:
                print(f"    ※ 気象庁の1回あたりデータ量上限を超えないよう自動分割しています")
            if discontinued:
                print(f"  ⚠ 廃止済み地点が {len(discontinued)} 件（期間により空データの場合あり）")
            if mismatch:
                ex = "、".join(mismatch[:5]) + ("…" if len(mismatch) > 5 else "")
                print(f"  ⚠ 選んだ項目を観測していない可能性のある地点 {len(mismatch)} 件（例: {ex}）")
            print()

            raw_dir = Path("jma_raw")
            raw_dir.mkdir(exist_ok=True)
            merged_dir = Path("jma_data")
            merged_dir.mkdir(exist_ok=True)

            session = requests.Session()
            session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
            session.get(ROOT_URL, timeout=30)
            time.sleep(sleep_sec)

            n_ok = n_skip = n_fail = 0
            failures: List[str] = []

            for s in stations:
                print(f"取得中: {s.name} ({s.station_num()})")
                saved_paths: List[Path] = []
                for spec_inter, ymd, suffix in plan:
                    fname = f"{s.name}_{period_label}_{suffix}.csv"
                    out_path = raw_dir / fname
                    if out_path.exists() and out_path.stat().st_size > 0:
                        print(f"  スキップ(取得済): {fname}")
                        saved_paths.append(out_path)
                        n_skip += 1
                        continue
                    try:
                        content = fetch_data(session, s.station_num(), ymd, period_code,
                                             element_codes, spec_inter, sleep_sec)
                        save_csv(content, out_path)
                        saved_paths.append(out_path)
                        print(f"  保存: {fname}")
                        n_ok += 1
                    except Exception as e:
                        print(f"  [エラー] {fname}: {e}")
                        failures.append(f"{fname}: {e}")
                        n_fail += 1
                    time.sleep(sleep_sec)

                # 地点ごとに結合
                if merge_cb.value and saved_paths:
                    merged_path = merged_dir / f"{s.name}_{period_label}.csv"
                    try:
                        merge_station_files(saved_paths, merged_path)
                    except Exception as e:
                        print(f"  [結合エラー] {s.name}: {e}")

            print()
            print("── 結果サマリ ──")
            print(f"  成功: {n_ok} / スキップ(取得済): {n_skip} / 失敗: {n_fail}")
            if failures:
                print("  失敗した項目:")
                for f in failures:
                    print(f"    - {f}")

            # ZIP化：結合ONなら結合ファイル、OFFなら生ファイル
            target_dir = merged_dir if merge_cb.value else raw_dir
            has_files = any(target_dir.iterdir())
            if has_files:
                print()
                print("ZIPにまとめてダウンロードします...")
                shutil.make_archive("jma_data_result", "zip", target_dir)
                files.download("jma_data_result.zip")
            else:
                print("保存できたファイルがないため、ZIPは作成しませんでした。")
        finally:
            run_btn.disabled = False


run_btn.on_click(on_run)

form = widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>気象庁データ 一括ダウンロード</h3>"
                 "<div style='color:#555;font-size:13px'>① 地点リストを渡し（貼り付け or CSV選択）、②以降を選んで「ダウンロード開始」を押してください。</div>"),
    _row(widgets.Label("① 地点CSV", layout=widgets.Layout(width="96px")), up),
    paste_area,
    mode_dd,
    period_dd,
    elem_sel,
    period_desc,
    start_row,
    end_row,
    sleep_dd,
    merge_cb,
    run_btn,
    out,
], layout=widgets.Layout(max_width="640px"))

display(form)
